# **Bioinformatics PDL1 Project**

---

## **ChEMBL Database**

The [*ChEMBL Database*](https://www.ebi.ac.uk/chembl/) is a database that contains curated bioactivity data of more than 2.5 million compounds. It is compiled from more than 92,100 documents, 1.7 million assays and the data spans 16,000 targets and 2,100 cells and 48,800 indications.
[Data of May 07, 2025; ChEMBL version 35].

## **Installing libraries**

Install the ChEMBL web service package so that we can retrieve bioactivity data from the ChEMBL Database.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
! pip install chembl_webresource_client

## **Importing libraries**

In [ ]:
# Import necessary libraries
import pandas as pd
from chembl_webresource_client.new_client import new_client

### **Target search for PDL1**

In [ ]:
# Target search for coronavirus
target = new_client.target
target_query = target.search('CHEMBL4523993')
targets = pd.DataFrame.from_dict(target_query)
targets

:### **Select and retrieve bioactivity data for *PDL1***

---



We will assign the 1st entry (which corresponds to the target protein, *PDL1*) to the ***selected_target*** variable

In [ ]:
selected_target = targets.target_chembl_id[0]
selected_target

Here, we will retrieve only bioactivity data for *PDL1* (CHEMBL4523993) that are reported as pChEMBL values.

In [ ]:
activity = new_client.activity
res = activity.filter(target_chembl_id=selected_target).filter(standard_type="IC50")

In [ ]:
df = pd.DataFrame.from_dict(res)

In [ ]:
df

## **Handling missing data**
If any compounds has missing value for the **standard_value** and **canonical_smiles** column then drop it.

In [ ]:
df2 = df[df.standard_value.notna()]
df2 = df2[df.canonical_smiles.notna()]
df2

In [ ]:
len(df2.canonical_smiles.unique())

In [ ]:
df2_nr = df2.drop_duplicates(['canonical_smiles'])
df2_nr

In [ ]:
print(type(df2_nr))

In [ ]:
df2_nr.to_csv('df2_nr.csv', index=False)

## **Data pre-processing of the bioactivity data**

### **Combine the 3 columns (molecule_chembl_id,canonical_smiles,standard_value) and bioactivity_class into a DataFrame**

In [ ]:
selection = ['molecule_chembl_id','canonical_smiles','standard_value']
df3 = df2_nr[selection]
df3

Saves dataframe to CSV file

In [ ]:
df3.to_csv('df3.csv', index=False)

### **Labeling compounds as either being active, inactive or intermediate**
The bioactivity data is in the IC50 unit. Compounds having values of less than 100 nM will be considered to be **active** while those greater than 1,000 nM will be considered to be **inactive**.

In [ ]:
df4 = pd.read_csv('df3.csv')

In [ ]:
df4

In [ ]:
bioactivity_threshold = []
for i in df4.standard_value:
  if float(i) >= 100:
    bioactivity_threshold.append("inactive")
  else float(i) <= 100:
    bioactivity_threshold.append("active")

In [ ]:
bioactivity_class = pd.Series(bioactivity_threshold, name='class')
df5 = pd.concat([df4, bioactivity_class], axis=1)
df5

Saves dataframe to CSV file

In [ ]:
df5.to_csv('df5.csv', index=False)

# **Exploratory Data Analysis**
---

## **Install conda and rdkit**

In [ ]:
! wget https://repo.anaconda.com/miniconda/Miniconda3-py37_4.8.2-Linux-x86_64.sh
! chmod +x Miniconda3-py37_4.8.2-Linux-x86_64.sh
! bash ./Miniconda3-py37_4.8.2-Linux-x86_64.sh -b -f -p /usr/local
! conda install -c rdkit rdkit -y
import sys
sys.path.append('/usr/local/lib/python3.7/site-packages/')

## **Load bioactivity data**

In [ ]:
df5_no_smiles = df4.drop(columns='canonical_smiles')

In [ ]:
df4_no_smiles

In [ ]:
smiles = []

for i in df4.canonical_smiles.tolist():
  cpd = str(i).split('.')
  cpd_longest = max(cpd, key = len)
  smiles.append(cpd_longest)

smiles = pd.Series(smiles, name = 'canonical_smiles')

In [ ]:
df5_clean_smiles = pd.concat([df4_no_smiles,smiles], axis=1)
df5_clean_smiles

In [ ]:
df1 = df5_clean_smiles.dropna()

In [ ]:
df1


## **Calculate Lipinski descriptors**
Christopher Lipinski, a scientist at Pfizer, came up with a set of rule-of-thumb for evaluating the **druglikeness** of compounds. Such druglikeness is based on the Absorption, Distribution, Metabolism and Excretion (ADME) that is also known as the pharmacokinetic profile. Lipinski analyzed all orally active FDA-approved drugs in the formulation of what is to be known as the **Rule-of-Five** or **Lipinski's Rule**.

The Lipinski's Rule stated the following:
* Molecular weight < 500 Dalton
* Octanol-water partition coefficient (LogP) < 5
* Hydrogen bond donors < 5
* Hydrogen bond acceptors < 10

### **Import libraries**

In [ ]:
!pip install rdkit-pypi


In [ ]:
from rdkit.Chem import Descriptors, Lipinski


### **Calculate descriptors**

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
import numpy as np
import pandas as pd

def lipinski(smiles, verbose=False):
    moldata = []
    for elem in smiles:
        mol = Chem.MolFromSmiles(elem)
        moldata.append(mol)

    baseData = np.arange(1, 1)
    i = 0
    for mol in moldata:
        desc_MolWt = Descriptors.MolWt(mol)
        desc_MolLogP = Descriptors.MolLogP(mol)
        desc_NumHDonors = Lipinski.NumHDonors(mol)
        desc_NumHAcceptors = Lipinski.NumHAcceptors(mol)
        desc_NumRotatableBonds = Lipinski.NumRotatableBonds(mol)


        # Calculate TPSA
        desc_TPSA = Descriptors.TPSA(mol)

        row = np.array([desc_MolWt,
                        desc_MolLogP,
                        desc_NumHDonors,
                        desc_NumHAcceptors,
                        desc_NumRotatableBonds,
                        desc_TPSA  # Add TPSA to the array
                        ])

        if i == 0:
            baseData = row
        else:
            baseData = np.vstack([baseData, row])
        i += 1

    columnNames = ["MW", "LogP", "NumHDonors", "NumHAcceptors", "NumRotatableBonds", "TPSA"]
    descriptors = pd.DataFrame(data=baseData, columns=columnNames)

    return descriptors

In [ ]:
df_lipinski = lipinski(df.canonical_smiles)
df_lipinski

### **Combine DataFrames**

Let's take a look at the 2 DataFrames that will be combined.

In [ ]:
df_lipinski

Now, let's combine the 2 DataFrame

In [ ]:
df_combined = pd.concat([df,df_lipinski], axis=1)

In [ ]:
df_combined

In [ ]:
df_combined.to_csv('Lipinski.csv')

### **Convert IC50 to pIC50**
To allow **IC50** data to be more uniformly distributed, we will convert **IC50** to the negative logarithmic scale which is essentially **-log10(IC50)**.

This custom function pIC50() will accept a DataFrame as input and will:
* Take the IC50 values from the ``standard_value`` column and converts it from nM to M by multiplying the value by 10$^{-9}$
* Take the molar value and apply -log10
* Delete the ``standard_value`` column and create a new ``pIC50`` column

In [ ]:
import numpy as np

def pIC50(input):
    pIC50 = []

    for i in input['standard_value_norm']:
        molar = i*(10**-9) # Converts nM to M
        pIC50.append(-np.log10(molar))

    input['pIC50'] = pIC50
    x = input.drop('standard_value_norm', axis=1)

    return x


Point to note: Values greater than 100,000,000 will be fixed at 100,000,000 otherwise the negative logarithmic value will become negative.

In [ ]:
df_combined.standard_value.describe()

In [ ]:
-np.log10( (10**-9)* 100000000 )

In [ ]:
-np.log10( (10**-9)* 10000000000 )

We will first apply the norm_value() function so that the values in the standard_value column is normalized.

In [ ]:

def norm_value(input):
    norm = []

    for i in input['standard_value']:
        if i > 100000000:
          i = 100000000
        norm.append(i)

    input['standard_value_norm'] = norm
    x = input.drop('standard_value', axis=1)

    return x

In [ ]:
df_norm = norm_value(df_combined)
df_norm

In [ ]:
df_norm.standard_value_norm.describe()

In [ ]:
df_final = pIC50(df_norm)
df_final

In [ ]:
df_final.to_csv('PDL1_pIC50.csv')

In [ ]:
df_final.pIC50.describe()

Let's write this to CSV file.

In [ ]:

# Map labels to groups
label_mapping = {
    'active': 'group1',
    'inactive': 'group2'
}

# Create a new column 'Group' based on the mapping
df_final['Groups'] = df_final['class'].map(label_mapping)
# Separate features and labels
features = df_final.drop(['Groups'], axis=1)  # Assuming 'class' and 'Group' are column names
labels = df_final['Groups']

# Now 'features' contains your input features, 'class' is preserved, and 'Group' contains the corresponding groups (0 or 1)
print(features)
print(labels)



## **Exploratory Data Analysis (Chemical Space Analysis) via Lipinski descriptors**

### **Import library**

In [ ]:
import pandas as pd

### **Frequency plot of the 2 bioactivity classes**

In [ ]:
df_final=pd.read_csv('/content/df_final_2classes.csv')

In [ ]:
df_final

In [ ]:
import seaborn as sns
sns.set(style='ticks')
import matplotlib.pyplot as plt
import pandas as pd # Make sure pandas is imported if not already

# Load the dataframe (assuming this was done in a previous cell)
# If not, uncomment the line below:
# df_final=pd.read_csv('chembl_zbi_classification.csv')


plt.figure(figsize=(8, 6))

# Assuming 'bioactivity_class' is the correct column name based on previous steps
sns.countplot(x='bioactivity_class', data=df_final, hue='bioactivity_class', edgecolor='black', palette=['#feb236', '#6b5b95'])

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('Frequency', fontsize=14, fontweight='bold')

# Save the plot as an SVG file
plt.savefig('plot_bioactivity_class.svg')

# Optionally, you can display the plot as well
# plt.show()

### **Scatter plot of MW versus LogP**

It can be seen that the 2 bioactivity classes are spanning similar chemical spaces as evident by the scatter plot of MW vs LogP.

In [ ]:
import numpy as np # Ensure numpy is imported for np.nan and np.ndarray
import pandas as pd # Ensure pandas is imported
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.scatterplot(x='MW', y='LogP', data=df_final, hue='bioactivity_class', size='pIC50', edgecolor='black', alpha=0.5, palette=['#feb236', '#6b5b95'])

plt.xlabel('MW in g/mol', fontsize=14, fontweight='bold', fontstyle='normal')
plt.ylabel('cLogP', fontsize=14, fontweight='bold', fontstyle='normal')
plt.legend(bbox_to_anchor=(0.01, 1), loc=2, borderaxespad=0, fontsize=9)
plt.savefig('plot_MW_vs_LogP.svg')

### **Box plots**

#### **pIC50 value**

In [ ]:
plt.figure(figsize=(8, 6))

sns.boxplot(x = 'bioactivity_class', y = 'pIC50', hue='bioactivity_class', data = df_final, palette=['#feb236', '#6b5b95'])

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('pIC50 value', fontsize=14, fontweight='bold')

plt.savefig('plot_ic50.svg')

**Statistical analysis | Mann-Whitney U Test**

In [ ]:
def mannwhitney(descriptor, verbose=False):
  # https://machinelearningmastery.com/nonparametric-statistical-significance-tests-in-python/
  from numpy.random import seed
  from numpy.random import randn
  from scipy.stats import mannwhitneyu

# seed the random number generator
  seed(1)

# actives and inactives
  selection = [descriptor, 'bioactivity_class']
  df = df_final[selection]
  active = df[df.bioactivity_class == 'active']
  active = active[descriptor]

  selection = [descriptor, 'bioactivity_class']
  df = df_final[selection]
  inactive = df[df.bioactivity_class == 'inactive']
  inactive = inactive[descriptor]

# compare samples
  stat, p = mannwhitneyu(active, inactive)
  #print('Statistics=%.3f, p=%.3f' % (stat, p))

# interpret
  alpha = 0.05
  if p > alpha:
    interpretation = 'Same distribution (fail to reject H0)'
  else:
    interpretation = 'Different distribution (reject H0)'

  results = pd.DataFrame({'Descriptor':descriptor,
                          'Statistics':stat,
                          'p':p,
                          'alpha':alpha,
                          'Interpretation':interpretation}, index=[0])
  filename = 'mannwhitneyu_' + descriptor + '.csv'
  results.to_csv(filename)

  return results

In [ ]:
mannwhitney('pIC50')

#### **MW**

In [ ]:
plt.figure(figsize=(8, 6))

sns.boxplot(x = 'bioactivity_class', y = 'MW', hue='bioactivity_class', data = df_final, palette=['#feb236', '#6b5b95'])

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('MW in g/mol', fontsize=14, fontweight='bold')

plt.savefig('plot_MW.svg')

In [ ]:
mannwhitney('MW')

#### **LogP**

In [ ]:
plt.figure(figsize=(8, 6))

sns.boxplot(x = 'bioactivity_class', y = 'LogP', hue='bioactivity_class', data = df_final, palette=['#feb236', '#6b5b95'])

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('cLogP', fontsize=14, fontweight='bold')

plt.savefig('plot_LogP.svg')

**Statistical analysis | Mann-Whitney U Test**

In [ ]:
mannwhitney('LogP')

#### **NumHDonors**

In [ ]:
plt.figure(figsize=(8, 6))

sns.boxplot(x = 'bioactivity_class', y = 'NumHDonors', hue='bioactivity_class', data = df_final, palette=['#feb236', '#6b5b95'])

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('nHD', fontsize=14, fontweight='bold')

plt.savefig('plot_NumHDonors.svg')

**Statistical analysis | Mann-Whitney U Test**

In [ ]:
mannwhitney('NumHDonors')

#### **NumHAcceptors**

In [ ]:
plt.figure(figsize=(8, 6))

sns.boxplot(x = 'bioactivity_class', y = 'NumHAcceptors', hue='bioactivity_class', data = df_final, palette=['#feb236', '#6b5b95'])

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('nHA', fontsize=14, fontweight='bold')


plt.savefig('plot_NumHAcceptors.svg')

In [ ]:
mannwhitney('NumHAcceptors')

In [ ]:
plt.figure(figsize=(8, 6))

sns.boxplot(x = 'bioactivity_class', y = 'TPSA', hue='bioactivity_class', data = df_final, palette=['#feb236', '#6b5b95'])

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('TPSA in Å²', fontsize=14, fontweight='bold')

plt.savefig('plot_TPSA.svg')

In [ ]:
mannwhitney('TPSA')

In [ ]:
plt.figure(figsize=(8, 6))

sns.boxplot(x = 'bioactivity_class', y = 'NumRotatableBonds', hue='bioactivity_class', data = df_final, palette=['#feb236', '#6b5b95'])

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('nRot', fontsize=14, fontweight='bold')

plt.savefig('plot_nRot.svg')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
mannwhitney('NumRotatableBonds')

# **BDescriptor Calculation and Dataset Preparation**
---

## **Download PaDEL-Descriptor**

In [ ]:
! apt-get update
! apt-get install -y openjdk-8-jre-headless

from padelpy import padeldescriptor

In [ ]:
! pip install padelpy

In [ ]:
! wget https://github.com/dataprofessor/padel/raw/main/fingerprints_xml.zip
! unzip fingerprints_xml.zip

In [ ]:
import glob
xml_files = glob.glob("*.xml")
xml_files.sort()
xml_files

In [ ]:
FP_list = ['AtomPairs2DCount',
 'AtomPairs2D',
 'EState',
 'CDKextended',
 'CDK',
 'CDKgraphonly',
 'KlekotaRothCount',
 'KlekotaRoth',
 'MACCS',
 'PubChem',
 'SubstructureCount',
 'Substructure']

In [ ]:
fp = dict(zip(FP_list, xml_files))
fp

In [ ]:
selection = ['smiles','ID']
df3_selection = df3[selection]
df3_selection.to_csv('molecule.smi', sep='\t', index=False, header=False)

In [ ]:
! cat molecule.smi | head -5

In [ ]:
! cat molecule.smi | wc -l

In [ ]:
descriptors = pd.read_csv(fingerprint_output_file)
descriptors

## **Calculate fingerprint descriptors**


In [ ]:
fingerprint = 'PubChem'

fingerprint_output_file = ''.join([fingerprint,'.csv']) #Substructure.csv
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='molecule.smi',
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='PubchemFingerprinter.xml',
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

In [ ]:
fp['PubChem']

### **Calculate PaDEL descriptors**

In [ ]:
df3_X = pd.read_csv('descriptors_output.csv')

## **Preparing the X and Y Data Matrices**

### **X data matrix**

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv('/content/mordred_descriptors.csv')

In [ ]:
df.columns

In [ ]:
df_7 = df.drop(columns=['compounds', 'MW', 'pIC50', 'Groups', 'LogP','NumHDonors', 'NumHAcceptors', 'NumRotatableBonds', 'TPSA'])

### **3.4. Remove high correlation columns and low variance features**

In [ ]:
import pandas as pd
from sklearn.feature_selection import VarianceThreshold

# Check if df_7 is already a DataFrame, if not, convert it
if not isinstance(df_7, pd.DataFrame):
    df_7 = pd.DataFrame(df_7)

# Correlation filtering
correlation_matrix = df_7.corr()
high_corr_columns = set()
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if abs(correlation_matrix.iloc[i, j]) > 0.90:
            colname = correlation_matrix.columns[i]
            high_corr_columns.add(colname)

df_7_filtered = df_7.drop(columns=high_corr_columns)
print(f"Removed columns: {high_corr_columns}")
print(f"Retained columns after correlation filtering: {df_7_filtered.columns.tolist()}")

# Variance thresholding
selection = VarianceThreshold(threshold=(.8 * (1 - .8)))
X = selection.fit_transform(df_7_filtered)

# Get and print retained column names after variance thresholding
# get_support returns a boolean array where True indicates the feature is retained
# use this to filter the column names
retained_columns_variance = df_7_filtered.columns[selection.get_support()].tolist()
print(f"Retained columns after variance thresholding: {retained_columns_variance}")

# Print shape
print(f"Shape of transformed data: {X.shape}")

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Assuming 'df_7' is your DataFrame containing the features to be scaled

# Standardization (Z-score scaling)
scaler_standard = StandardScaler()
df_scaled_standard = scaler_standard.fit_transform(df_7)
df_scaled_standard = pd.DataFrame(df_scaled_standard, columns=df_7.columns) # Convert back to DataFrame

print("DataFrame after Standardization:")
display(df_scaled_standard.head())

# Min-Max Scaling
scaler_minmax = MinMaxScaler()
df_scaled_minmax = scaler_minmax.fit_transform(df_7)
df_scaled_minmax = pd.DataFrame(df_scaled_minmax, columns=df_7.columns) # Convert back to DataFrame

print("\nDataFrame after Min-Max Scaling:")
display(df_scaled_minmax.head())

In [ ]:
df_7=df_scaled_standard

In [ ]:
df_7

In [ ]:
X.to_csv('X.csv', index=True)

In [ ]:
X

In [ ]:
Y=df['Groups']

In [ ]:
Y.to_csv('Y.csv', index=True)

## **Combining X and Y variable**

## **1. Import libraries**

### **3.3. Let's examine the data dimension**

## **4. Data split (80/20 ratio)**

In [ ]:
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split # Import train_test_split

X, Y = shuffle(X, Y, random_state=42)  # Shuffle X and Y together

# ... then continue with train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape, Y_train.shape

In [ ]:
X_test.shape, Y_test.shape

In [ ]:
# Convert X_test to a pandas DataFrame before calling to_csv
X_test = pd.DataFrame(X_test)  # Create a DataFrame from the NumPy array
X_test.to_csv('X_test.csv', index=True)  # Now you can use to_csv

In [ ]:
X_train.to_csv('X_train.csv', index=False)

In [ ]:
X_test.to_csv('X_test.csv', index=False)

In [ ]:
Y_train.to_csv('Y_train.csv', index=False)

In [ ]:
Y_test.to_csv('Y_test.csv', index=False)

# **Model Building**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
import lightgbm as lgb

In [ ]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Define the hyperparameter grid
param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.1, 1.0],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5, 10]
}

# Initialize GradientBoostingClassifier and GridSearchCV
gbc = GradientBoostingClassifier(random_state=42)
grid_search = GridSearchCV(gbc, param_grid=param_grid, cv=10, scoring='accuracy', verbose=2)

# Fit the GridSearchCV object to the training data
grid_search.fit(X_train, Y_train)

# Get the best model and its parameters
best_gbc = grid_search.best_estimator_
print("Best parameters found: ", grid_search.best_params_)

In [ ]:
!pip install lightgbm
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
# Define the hyperparameter grid
param_grid = {
    'max_depth': [3, 5, 9],
    'learning_rate': [0.01, 0.1, 0.5],
    'n_estimators': [300, 500, 700]
}

# Initialize LGBMClassifier and GridSearchCV
lgbm = lgb.LGBMClassifier(random_state=42)
grid_search = GridSearchCV(lgbm, param_grid=param_grid, cv=10, scoring='accuracy', verbose=2)

# Fit the GridSearchCV object to the training data
grid_search.fit(X_train, Y_train)

# Get the best model and its parameters
best_lgbm = grid_search.best_estimator_
print("Best parameters found: ", grid_search.best_params_)

# Make predictions on the test set
Y_pred = best_lgbm.predict(X_test)

# Evaluate the model's performance
accuracy = accuracy_score(Y_test, Y_pred)
print("Accuracy on test set: {:.2f}".format(accuracy))

In [ ]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Define the hyperparameter grid
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],  # Test different numbers of neighbors
    'weights': ['uniform', 'distance'],  # Test different weight functions
    'metric': ['euclidean', 'manhattan']  # Test different distance metrics
}

# Initialize KNeighborsClassifier and GridSearchCV
knn = KNeighborsClassifier()
grid_search = GridSearchCV(knn, param_grid=param_grid, cv=10, scoring='accuracy', verbose=2)

# Fit the GridSearchCV object to the training data
grid_search.fit(X_train, Y_train)

# Get the best model and its parameters
best_knn = grid_search.best_estimator_
print("Best parameters found: ", grid_search.best_params_)

# Make predictions on the test set
Y_pred = best_knn.predict(X_test)

# Evaluate the model's performance
accuracy = accuracy_score(Y_test, Y_pred)
print("Accuracy on test set: {:.2f}".format(accuracy))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Define the hyperparameter grid
param_grid = {
    'n_estimators': [100, 300, 500],
    'criterion': ['gini', 'entropy'],
    'max_depth': [5, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [3,5,7]
}

# Initialize ExtraTreesClassifier and GridSearchCV
etc = ExtraTreesClassifier(random_state=42)
grid_search = GridSearchCV(etc, param_grid=param_grid, cv=10, scoring='accuracy', verbose=2)

# Fit the GridSearchCV object to the training data
grid_search.fit(X_train, Y_train)

# Get the best model and its parameters
best_etc = grid_search.best_estimator_
print("Best parameters found: ", grid_search.best_params_)

# Make predictions on the test set
Y_pred = best_etc.predict(X_test)

# Evaluate the model's performance
accuracy = accuracy_score(Y_test, Y_pred)
print("Accuracy on test set: {:.2f}".format(accuracy))

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Define the hyperparameter grid
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [3,5,7]
}

# Initialize DecisionTreeClassifier and GridSearchCV
dtc = DecisionTreeClassifier(random_state=42)
grid_search = GridSearchCV(dtc, param_grid=param_grid, cv=10, scoring='accuracy', verbose=2)

# Fit the GridSearchCV object to the training data
grid_search.fit(X_train, Y_train)

# Get the best model and its parameters
best_dtc = grid_search.best_estimator_
print("Best parameters found: ", grid_search.best_params_)

# Make predictions on the test set
Y_pred = best_dtc.predict(X_test)

# Evaluate the model's performance
accuracy = accuracy_score(Y_test, Y_pred)
print("Accuracy on test set: {:.2f}".format(accuracy))

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
import numpy as np #Added this line to import the numpy module
np.random.seed(42)

# Define the hyperparameter grid
param_grid = {
    'penalty': ['l1', 'l2'],
    'C': np.logspace(-4, 4, 20),
    'solver': ['liblinear', 'sag', 'saga', 'newton-cg', 'lbfgs'],
    'max_iter':[1000]
}

# Initialize LogisticRegression and GridSearchCV
logreg = LogisticRegression(random_state=42, max_iter=1000)
grid_search = GridSearchCV(logreg, param_grid=param_grid, cv=10, scoring='accuracy', verbose=2)

# Fit the GridSearchCV object to the training data
grid_search.fit(X_train, Y_train)

# Get the best model and its parameters
best_logreg = grid_search.best_estimator_
print("Best parameters found: ", grid_search.best_params_)

# Make predictions on the test set
Y_pred = best_logreg.predict(X_test)

# Evaluate the model's performance
accuracy = accuracy_score(Y_test, Y_pred)
print("Accuracy on test set: {:.2f}".format(accuracy))

In [ ]:
param_grid = {'C': [0.1, 1, 10],
              'gamma': [0.1, 1, 10]}
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

grid_search = GridSearchCV(SVC(kernel='rbf', random_state=42),
                           param_grid, cv=10)
grid_search.fit(X_train, Y_train)
print("Best parameters found: ", grid_search.best_params_)
best_svm = grid_search.best_estimator_
# Make predictions on the test set
Y_pred = best_svm.predict(X_test)

# Evaluate the model's performance
accuracy = accuracy_score(Y_test, Y_pred)
print("Accuracy on test set: {:.2f}".format(accuracy))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.metrics import accuracy_score # Import accuracy_score here

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Define the hyperparameter grid for GaussianProcessClassifier
param_grid = {
    'kernel': [RBF(length_scale=length_scale)
               for length_scale in [0.1, 1.0, 10.0]],
    'kernel__length_scale_bounds': [(1e-5, 1e3)]
}

# Initialize GaussianProcessClassifier and GridSearchCV
gpc = GaussianProcessClassifier(random_state=42)
grid_search = GridSearchCV(gpc, param_grid, cv=10)

# Fit the GridSearchCV object to the training data
grid_search.fit(X_train, Y_train)

# Get the best model and its parameters
best_gpc = grid_search.best_estimator_
print("Best parameters found: ", grid_search.best_params_)
# Make predictions on the test set
Y_pred = best_gpc.predict(X_test)

# Evaluate the model's performance
accuracy = accuracy_score(Y_test, Y_pred)
print("Accuracy on test set: {:.2f}".format(accuracy))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Define the hyperparameter grid for RandomForestClassifier
param_grid = {
    'n_estimators': [100, 300, 500,700],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [3,5,7],
    'criterion': ['gini', 'entropy']
}

# Initialize RandomForestClassifier and GridSearchCV
rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid=param_grid, cv=5, scoring='accuracy', verbose=2)

# Fit the GridSearchCV object to the training data
grid_search.fit(X_train, Y_train)

# Get the best model and its parameters
best_rf = grid_search.best_estimator_
print("Best parameters found: ", grid_search.best_params_)

# Make predictions on the test set
Y_pred = best_rf.predict(X_test)

# Evaluate the model's performance
accuracy = accuracy_score(Y_test, Y_pred)
print("Accuracy on test set: {:.2f}".format(accuracy))

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
import lightgbm as lgb
import numpy as np

In [ ]:
#Mordred
names = ["Nearest_Neighbors", "SVC", "Gaussian_Process","Gradient_Boosting", "DecisionTreeClassifier", "ExtraTreesClassifier", "RandomForestClassifier", "LogisticRegression", "lightgbm","Naive_Bayes"]

from sklearn.gaussian_process.kernels import RBF # Import RBF here

classifiers = [
    KNeighborsClassifier(metric='manhattan', n_neighbors=3, weights='distance'),
    SVC(C=10, gamma=0.1),
    GaussianProcessClassifier(kernel=RBF(length_scale=0.1),random_state=42),
    GradientBoostingClassifier(learning_rate=1, max_depth=5, min_samples_split=10, n_estimators=300, random_state=42),
    DecisionTreeClassifier(criterion='gini', max_depth=10, max_features=7, min_samples_leaf=1, min_samples_split=10, random_state=42),
    ExtraTreesClassifier(criterion='gini', max_depth=10, max_features=7, min_samples_leaf=1, min_samples_split=5, n_estimators=300, random_state=42),
    RandomForestClassifier(criterion='entropy', max_depth=15, max_features=5, min_samples_leaf=1, min_samples_split=2, n_estimators=500, random_state=42),
    LogisticRegression(C=float(1.623776739188721), max_iter=1000, penalty='l1', solver='saga', random_state=42),
    lgb.LGBMClassifier(learning_rate=0.1, max_depth=5, n_estimators=300, random_state=42),
    GaussianNB()]

In [ ]:
#PubChem
names = ["Nearest_Neighbors", "SVC", "Gaussian_Process","Gradient_Boosting", "DecisionTreeClassifier", "ExtraTreesClassifier", "RandomForestClassifier", "LogisticRegression", "lightgbm","Naive_Bayes"]

from sklearn.gaussian_process.kernels import RBF # Import RBF here
classifiers = [
    KNeighborsClassifier(metric='euclidean', n_neighbors=3, weights='distance'),
    SVC(C=1, gamma=0.1),
    GaussianProcessClassifier(kernel=RBF(length_scale=0.1),random_state=42),
    GradientBoostingClassifier(learning_rate=0.1, max_depth=3, min_samples_split=5, n_estimators=100, random_state=42),
    DecisionTreeClassifier(criterion='gini', max_depth=15, max_features=3, min_samples_leaf=1, min_samples_split=2, random_state=42),
    ExtraTreesClassifier(criterion='gini', max_depth=15, max_features=7, min_samples_leaf=1, min_samples_split=10, n_estimators=500, random_state=42),
    RandomForestClassifier(criterion='gini', max_depth=10, max_features=5, min_samples_leaf=1, min_samples_split=2, n_estimators=300, random_state=42),
    LogisticRegression(C=float(4.281332398719396), max_iter=1000, penalty='l2', solver='liblinear', random_state=42),
    lgb.LGBMClassifier(learning_rate=0.01, max_depth=9, n_estimators=500, random_state=42),
    GaussianNB()]



In [ ]:
#MACCS
names = ["Nearest_Neighbors", "SVC", "Gaussian_Process","Gradient_Boosting", "DecisionTreeClassifier", "ExtraTreesClassifier", "RandomForestClassifier", "LogisticRegression", "lightgbm","Naive_Bayes"]

from sklearn.gaussian_process.kernels import RBF # Import RBF here
classifiers = [
    KNeighborsClassifier(metric='manhattan', n_neighbors=3, weights='distance'),
    SVC(C=10, gamma=0.1),
    GaussianProcessClassifier(kernel=RBF(length_scale=0.1),random_state=42),
    GradientBoostingClassifier(learning_rate=0.01, max_depth=5, min_samples_split=2, n_estimators=300, random_state=42),
    DecisionTreeClassifier(criterion='entropy', max_depth=10, max_features=7, min_samples_leaf=2, min_samples_split=5, random_state=42),
    ExtraTreesClassifier(criterion='entropy', max_depth=10, max_features=7, min_samples_leaf=1, min_samples_split=2, n_estimators=300, random_state=42),
    RandomForestClassifier(criterion='entropy', max_depth=10, max_features=5, min_samples_leaf=1, min_samples_split=2, n_estimators=300, random_state=42),
    LogisticRegression(C=float(0.615848211066026), max_iter=1000, penalty='l1', solver='saga', random_state=42),
    lgb.LGBMClassifier(learning_rate=0.5, max_depth=3, n_estimators=300, random_state=42),
    GaussianNB()]

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, matthews_corrcoef
from sklearn.model_selection import cross_val_predict
import pandas as pd

# Assuming 'names' is your list of classifier names
# Assuming 'classifiers' is your list of classifiers
# Assuming 'X_train', 'Y_train', 'X_test', 'Y_test' are your training and test data

# Evaluate each classifier on Train, CV, and Test sets
results = []

for name, clf in zip(names, classifiers):
    # Training set evaluation
    clf.fit(X_train, Y_train)
    Y_train_pred = clf.predict(X_train)

    accuracy_train = accuracy_score(Y_train, Y_train_pred)

    mcc_train = matthews_corrcoef(Y_train, Y_train_pred)

    # Cross-validation
    Y_cv_pred = cross_val_predict(clf, X_train, Y_train, cv=10)

    accuracy_cv = accuracy_score(Y_train, Y_cv_pred)

    mcc_cv = matthews_corrcoef(Y_train, Y_cv_pred)

    # Test set evaluation
    Y_test_pred = clf.predict(X_test)

    accuracy_test = accuracy_score(Y_test, Y_test_pred)

    mcc_test = matthews_corrcoef(Y_test, Y_test_pred)

    results.append({
        "Classifier": name,
        "Accuracy_Train": accuracy_train,

        "MCC_Train": mcc_train,
        "Accuracy_CV": accuracy_cv,

        "MCC_CV": mcc_cv,
        "Accuracy_Test": accuracy_test,

        "MCC_Test": mcc_test
    })

# Display the results
results_df = pd.DataFrame(results)
print(results_df)


In [ ]:
!pip install tabulate
from tabulate import tabulate

# Assuming 'results_df' is your DataFrame with the evaluation results
print(tabulate(results_df, headers='keys', tablefmt='psql'))

In [ ]:
results_df.to_csv('results.csv', index=True)

In [ ]:
train = pd.concat([pd.DataFrame(X_train), Y_train.reset_index(drop=True)], axis=1)
test = pd.concat([pd.DataFrame(X_test), Y_test.reset_index(drop=True)], axis=1)

train['model'] = "Train"
test['model'] = "Test"

In [ ]:
train['model'] = "Train"
test['model'] = "Test"

In [ ]:
frames = [train,test]
pca = pd.concat(frames)
pca.head(2)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator
from pandas.plotting import scatter_matrix # Changed the import statement to use pandas.plotting
# Plot the Figures Inline
%matplotlib inline


def PCA_plot(data):
    # PCA's components graphed in 2D
    # Apply Scaling
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler
    from matplotlib import pyplot as plt
    data_pca = data

    # Apply Scaling
    X = data_pca.drop('model', axis=1).values # Use .values instead of as_matrix()
    y = data_pca['model'].values

    # Formatting
    target_names = ['Train','Test']
    colors = ['#feb236', '#6b5b95']

    # 2 Components PCA

    fig=plt.figure(2, figsize=(8, 6)) # Adjust figure size if needed

    pca = PCA(n_components=2)
    X_std = StandardScaler().fit_transform(X)
    X_r = pca.fit_transform(X_std)

    for color, i, target_name in zip(colors, ['Train','Test'], target_names):
        plt.scatter(X_r[y == i, 0], X_r[y == i, 1],
                    color=color,

                    label=target_name)
    plt.xlabel('PC1',  fontstyle= "normal", fontsize=14, fontweight='bold')
    plt.ylabel('PC2',  fontstyle= "normal", fontsize=14, fontweight='bold')


    plt.grid(False) #remove grid in 2D plot

    # Add outline around axes
    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_linewidth(1)
        spine.set_color('black')

    # Legend with box and black outline
    legend = plt.legend(loc='best', scatterpoints=1)
    legend.get_frame().set_linewidth(2)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    plt.tick_params(direction="out", labelsize=12)
    plt.savefig('applicability_domain.svg')
PCA_plot(pca)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator
from pandas.plotting import scatter_matrix
# Plot the Figures Inline
%matplotlib inline

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from matplotlib import pyplot as plt
import pandas as pd # Ensure pandas is imported

def PCA_plot(data):
    # PCA's components graphed in 2D
    # Apply Scaling

    # Apply Scaling
    # Drop any non-numeric columns (like 'bioactivity_class' if it exists) before scaling
    # Ensure 'model' is the only column dropped if it's a label/identifier
    X = data.drop('model', axis=1).select_dtypes(include=np.number).values # Select only numeric columns
    y = data['model'].values

    # Formatting
    # Ensure target_names and colors match the unique values in y
    unique_labels = np.unique(y)
    target_names = unique_labels
    # Updated colors based on your request
    colors = ['#feb236' if label == 'Train' else '#6b5b95' for label in unique_labels]


    # 2 Components PCA

    fig=plt.figure(2, figsize=(8, 6)) # Adjust figure size if needed

    pca = PCA(n_components=2)
    # Check if X is empty before scaling
    if X.shape[0] == 0 or X.shape[1] == 0:
         print("Error: Input data for PCA is empty or has no features.")
         return # Exit the function if data is empty


    X_std = StandardScaler().fit_transform(X)
    X_r = pca.fit_transform(X_std)

    # Iterate through unique labels and colors to plot
    for color, target_name in zip(colors, target_names):
        plt.scatter(X_r[y == target_name, 0], X_r[y == target_name, 1],
                    color=color,
                    label=target_name,
                    alpha=0.5,          # Added alpha=0.5
                    edgecolor='black')  # Added edgecolor='black'
    plt.xlabel('PC1',  fontstyle= "normal", fontsize=14, fontweight='bold')
    plt.ylabel('PC2',  fontstyle= "normal", fontsize=14, fontweight='bold')


    plt.grid(False) #remove grid in 2D plot

    # Add outline around axes
    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_linewidth(1)
        spine.set_color('black')

    # Legend with box and black outline
    legend = plt.legend(loc='best', scatterpoints=1)
    legend.get_frame().set_linewidth(2)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    plt.tick_params(direction="out", labelsize=12)
    plt.savefig('applicability_domain.svg')

# Ensure the 'pca' DataFrame passed to PCA_plot only contains numeric features and the 'model' column.
# Assuming 'df3_X' is the DataFrame containing the original data before splitting
# and 'df_7_filtered' was the numeric feature matrix used for model training.
# We need to combine the numeric features from the original data with the train/test split labels.

# Recreate the train and test dataframes used for PCA plot, ensuring they only contain numeric features and the 'model' column
train_pca = pd.DataFrame(X_train, columns=[f'feature_{i}' for i in range(X_train.shape[1])])
train_pca['model'] = "Train"

test_pca = pd.DataFrame(X_test, columns=[f'feature_{i}' for i in range(X_test.shape[1])])
test_pca['model'] = "Test"

# Concatenate the train and test dataframes for PCA plotting
pca_combined = pd.concat([train_pca, test_pca])

# Call the PCA_plot function with the clean data
PCA_plot(pca_combined)

In [ ]:
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from sklearn.decomposition import PCA

# Assuming 'X_train', 'Y_train', 'X_test', 'Y_test' are your training and test data

# Train your best model (Extra Trees, for example)
best_model = LGBMClassifier()
best_model.fit(X_train, Y_train)

# Applying PCA to training data
pca_train = PCA(n_components=2)
X_train_pca = pca_train.fit_transform(X_train)

# Applying PCA to test data
pca_test = PCA(n_components=2)
X_test_pca = pca_test.fit_transform(X_test)

# Visualize PCA of training data
plt.figure(figsize=(8, 6))
plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], color='blue', label='Training-set')

plt.xlabel('PC1')
plt.ylabel('PC2')

# Overlay PCA of test data on the same plot
plt.scatter(X_test_pca[:, 0], X_test_pca[:, 1], color='red', label='Test-set')
plt.legend()

plt.show()
plt.savefig('PCA.svg')


In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns # Import seaborn

# Assuming df_final is your DataFrame
features = ['MW', 'LogP', 'NumHDonors', 'NumHAcceptors', 'NumRotatableBonds', 'TPSA']

# Separate features and labels
X = df_final[features]
y = df_final['bioactivity_class']

# Standardize the data (important for PCA)
X_std = StandardScaler().fit_transform(X)

# Perform PCA
pca = PCA(n_components=2)  # You can change the number of components as needed
principal_components = pca.fit_transform(X_std)

# Create a new DataFrame with the principal components and relevant columns
pca_df = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2'])
# Ensure the 'bioactivity_class' and 'pIC50' columns are also in pca_df
pca_df['bioactivity_class'] = df_final['bioactivity_class'].values
pca_df['pIC50'] = df_final['pIC50'].values

# Diagnostic print statement (optional)
print("Unique values in 'bioactivity_class' column:", pca_df['bioactivity_class'].unique())

# Plot the PCA results
plt.figure(figsize=(8, 6))

# Define the colors for each bioactivity class
colors = {'active': '#6b5b95', 'inactive': '#feb236'}

for bioactivity_class, color in colors.items():
    # Filter the data for the current bioactivity class
    class_data = pca_df[pca_df['bioactivity_class'] == bioactivity_class]
    # Plot the data for this class with the specified color
    if not class_data.empty:
        plt.scatter(class_data['PC1'], class_data['PC2'], alpha=0.4, color=color, label=bioactivity_class, edgecolor='black')
    else:
        print(f"Warning: No data found for bioactivity_class '{bioactivity_class}' to plot.")


plt.xlabel('PC1', fontsize=14, fontweight='bold')
plt.ylabel('PC2', fontsize=14, fontweight='bold')
plt.legend()

# --- Add this line to save the figure before showing it ---
plt.savefig('PCA.svg', format='svg')

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import scipy.stats

# Assuming 'df_final' is your DataFrame containing the relevant columns

# List of columns
columns_of_interest = ["MW", "LogP", "NumHAcceptors", "NumHDonors", "NumRotatableBonds", "TPSA"]

# Dictionary to store results
summary_statistics = {}

# Separate data for group1 and group2
for group_name in df_final['bioactivity_class'].unique():
    summary_df = df_final[df_final['bioactivity_class'] == group_name][columns_of_interest].describe().transpose()

    # Calculate skewness and kurtosis
    skewness_values = df_final[df_final['bioactivity_class'] == group_name][columns_of_interest].skew()
    kurtosis_values = df_final[df_final['bioactivity_class'] == group_name][columns_of_interest].kurt()

    # Add skewness and kurtosis to the summary DataFrame
    summary_df['skew'] = skewness_values
    summary_df['kurt'] = kurtosis_values

    # Add p-values to the table
    p_values = []
    for column in columns_of_interest:
        _, p_value = scipy.stats.ttest_ind(
            df_final[df_final['bioactivity_class'] == 'active'][column].dropna(),
            df_final[df_final['bioactivity_class'] == 'inactive'][column].dropna()
        )
        p_values.append(p_value)

    # Append p-values to the summary DataFrame
    summary_df['p-value'] = p_values

    # Store the summary DataFrame in the dictionary
    summary_statistics[group_name] = summary_df

# Display the results
for group_name, stats_df in summary_statistics.items():
    print(f"\nSummary statistics for {group_name}:")
    print(stats_df[['min', 'max', '50%', 'mean', 'skew', 'kurt', 'p-value']])

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, matthews_corrcoef, roc_auc_score,
                             f1_score, precision_score, recall_score,
                             confusion_matrix, log_loss)


def calculate_comprehensive_metrics(y_true, y_pred, y_pred_proba=None, class_label=1):
    """
    Calculate a comprehensive set of metrics for model evaluation.

    Parameters:
    y_true (array): True labels.
    y_pred (array): Predicted labels.
    y_pred_proba (array): Predicted probabilities for the positive class.
    class_label (int): The positive class label (default=1).

    Returns:
    dict: Dictionary containing all metrics.
    """

    metrics = {}

    # Metrics you already have
    metrics['Accuracy'] = accuracy_score(y_true, y_pred)
    metrics['MCC'] = matthews_corrcoef(y_true, y_pred)

    # Additional metrics requested by reviewer
    metrics['F1'] = f1_score(y_true, y_pred, pos_label=class_label)
    metrics['Precision'] = precision_score(y_true, y_pred, pos_label=class_label)
    metrics['Recall'] = recall_score(y_true, y_pred, pos_label=class_label)

    # Calculate specificity from confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    metrics['Specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0

    # AUC and Log Loss require probability estimates
    if y_pred_proba is not None and np.all(np.isin(y_true, [0, 1])) and y_pred_proba.ndim == 1:
        metrics['AUC'] = roc_auc_score(y_true, y_pred_proba)
        metrics['Log_Loss'] = log_loss(y_true, y_pred_proba)
    else:
        metrics['AUC'] = 'Not available (need probabilities)'
        metrics['Log_Loss'] = 'Not available (need probabilities)'

    return metrics


# Split the data into training and test sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Iterate through the trained classifiers and calculate metrics
results = [] # Initialize an empty list to store results
for name, clf in zip(names, classifiers):
    print(f"Metrics for {name}:")
    # Train the classifier before making predictions
    if name == "SVC":
      clf.fit(X_train, Y_train) # SVC needs probability=True during fit
    else:
      clf.fit(X_train, Y_train)

    y_pred = clf.predict(X_test)

    # Check if the classifier has predict_proba before calling it
    if hasattr(clf, 'predict_proba'):
        y_pred_proba = clf.predict_proba(X_test)[:, 1]  # Probabilities for class 1
        test_metrics = calculate_comprehensive_metrics(Y_test, y_pred, y_pred_proba)
    else:
        # Handle classifiers that don't have predict_proba
        test_metrics = calculate_comprehensive_metrics(Y_test, y_pred)

    results.append({"Classifier": name, **test_metrics}) # Append the metrics to the results list

    # Print in a nice format
    for metric, value in test_metrics.items():
        if isinstance(value, float):
            print(f"{metric}: {value:.4f}")
        else:
            print(f"{metric}: {value}")
    print("-" * 30)

In [ ]:
# 9. Save the trained model to a pickle file
import pickle
filename = 'RandomForestClassifier.pkl'

# Find the LogisticRegression classifier and train it with the training data
for name, clf in zip(names, classifiers):
    if name == "RandomForestClassifier":
        # Assuming X_train and Y_train are available from a previous cell
        clf.fit(X_train, Y_train) # Train the model before saving
        pickle.dump(clf, open(filename, 'wb'))
        print(f"Saved trained RandomForestClassifier model to {filename}")
        break

In [ ]:
# 9. Save the trained model to a pickle file
import pickle
filename = 'ExtraTreesClassifier.pkl'

# Find the LogisticRegression classifier and train it with the training data
for name, clf in zip(names, classifiers):
    if name == "ExtraTreesClassifier":
        # Assuming X_train and Y_train are available from a previous cell
        clf.fit(X_train, Y_train) # Train the model before saving
        pickle.dump(clf, open(filename, 'wb'))
        print(f"Saved trained ExtraTreesClassifier model to {filename}")
        break

In [ ]:
external_df_raw=pd.read_csv('/content/mordred_descriptors_external_dataset.csv')

In [ ]:
import pandas as pd
external_df_raw=pd.read_csv('/content/mordred_descriptors_external_dataset.csv')

# Apply the same filtering to the external dataset using 'retained_columns_variance'
# This assumes 'retained_columns_variance' is globally accessible from previous execution.

# Check if all retained_columns_variance are in external_df_raw
missing_cols_in_external = [col for col in retained_columns_variance if col not in external_df_raw.columns]

if missing_cols_in_external:
    print(f"Error: External dataset is missing columns required by the trained model: {missing_cols_in_external}")
    print("Please ensure your external dataset contains all features used for training.")
    # Handle this error appropriately, perhaps by filling missing columns with zeros or NaNs
    # For this fix, we'll add missing columns and fill with 0, or drop rows with NaN if appropriate.
    for col in missing_cols_in_external:
        external_df_raw[col] = 0 # Or np.nan, depending on how you want to handle it.

    # Reorder columns to match the training data
    new_data = external_df_raw[retained_columns_variance]
else:
    # Filter and reorder columns to match the training data
    new_data = external_df_raw[retained_columns_variance]

display(new_data.head())


In [ ]:
new_data.to_csv('new_data.csv', index=True)

In [ ]:
/content/random_forest_classifier_model.pkl

In [ ]:
import pickle
import pandas as pd

filename = 'RandomForestClassifier.pkl'
# Load the trained model instance
loaded_model = pickle.load(open(filename, 'rb'))

# Make predictions
predictions = loaded_model.predict(new_data)

# Map the predicted numerical labels (0 or 1) back to bioactivity classes ('active' or 'inactive')
# Assuming '0' corresponds to 'inactive' and '1' corresponds to 'active' based on previous mapping (df_final['Groups'])
# If your 'Groups' column used '0' for active and '1' for inactive, adjust this line.
# Based on the evaluation metrics for classifiers, it seems '1' corresponds to the positive class (active).
predicted_classes = ['active' if pred == 1 else 'inactive' for pred in predictions]

# Assuming 'external_df_raw' contains the original external data with a compound identifier
# Combine the compound identifiers with the predicted classes
if 'external_df_raw' in locals() and 'Name' in external_df_raw.columns:
    # Ensure the number of predictions matches the number of external compounds
    if len(predicted_classes) == len(external_df_raw):
        # Create a new DataFrame with the relevant info using the 'Name' column
        predictions_df = external_df_raw[['Name']].copy()
        predictions_df['Predicted_Bioactivity_Class'] = predicted_classes
        print("Predictions for external dataset:")
        display(predictions_df.head())
        # Optionally, save these predictions to a CSV
        predictions_df.to_csv('external_predictions.csv', index=False)
        print("Predictions saved to 'external_predictions.csv'")
    else:
        print("Warning: Number of predictions does not match the number of external compounds.")
        print("Predicted classes:", predicted_classes)
elif 'external_df_raw' in locals() and 'Name' not in external_df_raw.columns:
    print("Error: The 'external_df_raw' DataFrame does not contain a 'Name' column to identify compounds.")
    print("Predicted classes:", predicted_classes)
else:
    print("Error: Original external data (external_df_raw) not found in the kernel.")
    print("Predicted classes:", predicted_classes)

In [ ]:
import pickle
import pandas as pd

filename = 'ExtraTreesClassifier.pkl'
# Load the trained model instance
loaded_model = pickle.load(open(filename, 'rb'))

# Make predictions
predictions = loaded_model.predict(new_data)

predicted_classes = ['active' if pred == 1 else 'inactive' for pred in predictions]

# Assuming 'external_df_raw' contains the original external data with a compound identifier
# Combine the compound identifiers with the predicted classes
if 'external_df_raw' in locals() and 'Name' in external_df_raw.columns:
    # Ensure the number of predictions matches the number of external compounds
    if len(predicted_classes) == len(external_df_raw):
        # Create a new DataFrame with the relevant info using the 'Name' column
        predictions_df = external_df_raw[['Name']].copy()
        predictions_df['Predicted_Bioactivity_Class'] = predicted_classes
        print("Predictions for external dataset:")
        display(predictions_df.head())
        # Optionally, save these predictions to a CSV
        predictions_df.to_csv('external_predictions.csv', index=False)
        print("Predictions saved to 'external_predictions.csv'")
    else:
        print("Warning: Number of predictions does not match the number of external compounds.")
        print("Predicted classes:", predicted_classes)
elif 'external_df_raw' in locals() and 'Name' not in external_df_raw.columns:
    print("Error: The 'external_df_raw' DataFrame does not contain a 'Name' column to identify compounds.")
    print("Predicted classes:", predicted_classes)
else:
    print("Error: Original external data (external_df_raw) not found in the kernel.")
    print("Predicted classes:", predicted_classes)

In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score
import numpy as np

FILE_PATH = "/content/external-validation-pdl1-results.csv"
CMV_THRESHOLD = 3 # N >= 3 Active votes
OUTPUT_FILE = "consensus_results.csv"

def load_and_prepare_data(file_path, prediction_cols):
    """Loads data, filters for valid compounds, and standardizes class names."""
    print(f"Loading data from: {file_path}")
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return None

    # Filter to only rows with valid compound IDs (N=150)
    df_predictions = df[df['ID'].notna()].loc[:, ['ID'] + prediction_cols].copy()

    # Standardize the class names (lowercase and strip whitespace)
    for col in prediction_cols:
        # Fill any NaN with 'unknown' for safety, then standardize
        df_predictions[col] = df_predictions[col].astype(str).str.lower().str.strip().replace('nan', 'unknown')

    print(f"Data loaded successfully. Total compounds analyzed: {len(df_predictions)}")
    return df_predictions

# --- 3. COHEN'S KAPPA CALCULATION ---

def calculate_kappa_matrix(df_predictions, prediction_cols, short_names):
    """Calculates the Cohen's Kappa matrix for all model pairs."""
    print("\n--- Calculating Cohen's Kappa (κ) Matrix ---")
    kappa_matrix = pd.DataFrame(index=prediction_cols, columns=prediction_cols, dtype=float)

    num_models = len(prediction_cols)
    for i in range(num_models):
        for j in range(i, num_models):
            model1 = prediction_cols[i]
            model2 = prediction_cols[j]

            # Check for zero variance (Mordred_ET) which prevents Kappa calculation
            if df_predictions[model1].nunique() <= 1 or df_predictions[model2].nunique() <= 1:
                 # If one column is constant (e.g., all 'active'), Kappa is undefined or 0
                 kappa = 0.0
            else:
                kappa = cohen_kappa_score(df_predictions[model1], df_predictions[model2])

            kappa_matrix.loc[model1, model2] = kappa
            kappa_matrix.loc[model2, model1] = kappa # Matrix is symmetric

    # Rename columns/index for a cleaner output (as used in the manuscript)
    kappa_matrix = kappa_matrix.rename(index=short_names, columns=short_names)

    print("Cohen's Kappa (κ) Matrix for Inter-Model Agreement:")
    print(kappa_matrix.round(3))
    return kappa_matrix

# --- 4. CONSENSUS ANALYSIS ---

def perform_consensus_analysis(df_predictions, prediction_cols, short_names, threshold):
    """Calculates the Consensus Majority Vote (CMV) for each compound."""
    print(f"\n--- Performing Consensus Majority Vote (CMV) Analysis (Threshold N >= {threshold}) ---")

    # Rename columns in the working DataFrame for easier counting
    df_temp = df_predictions.rename(columns=short_names)
    short_cols = list(short_names.values())

    # Create a binary DataFrame: 1 for 'active', 0 otherwise
    df_binary = df_temp[short_cols].apply(
        lambda x: np.where(x == 'active', 1, 0)
    )

    # Calculate the sum of 'Active' votes for each compound
    df_temp['Active_Vote_Count'] = df_binary.sum(axis=1)

    # Determine the Final Consensus Class based on the threshold
    df_temp['Consensus_Class'] = np.where(
        df_temp['Active_Vote_Count'] >= threshold,
        'Active (CMV)',
        'Inactive (CMV)'
    )

    # Select and display the final consensus results
    final_consensus = df_temp[['ID', 'Active_Vote_Count', 'Consensus_Class'] + short_cols]

    print(f"\nConsensus Summary (Threshold >= {threshold} votes):")
    print(final_consensus['Consensus_Class'].value_counts())

    print("\nFirst 10 compounds and their vote breakdown:")
    print(final_consensus.head(10))

    return final_consensus

# --- 5. SAVE RESULTS ---

def save_results_to_csv(df, output_file_path):
    """Saves the final consensus results DataFrame to a CSV file."""
    try:
        df.to_csv(output_file_path, index=False)
        print(f"\nResults successfully saved to: {output_file_path}")
    except Exception as e:
        print(f"\nError saving results to CSV: {e}")

# --- 6. MAIN EXECUTION ---

if __name__ == '__main__':
    df_predictions = load_and_prepare_data(FILE_PATH, PREDICTION_COLS)

    if df_predictions is not None:
        # Calculate Kappa Matrix
        kappa_matrix = calculate_kappa_matrix(df_predictions, PREDICTION_COLS, SHORT_NAMES)

        # Perform Consensus Analysis
        final_consensus_results = perform_consensus_analysis(
            df_predictions, PREDICTION_COLS, SHORT_NAMES, CMV_THRESHOLD
        )

        # Save the final results to CSV
        save_results_to_csv(final_consensus_results, OUTPUT_FILE)